# 02 — SimBench Path A

**LLM-free distributional user simulator** — paper §4.1 의 primary external benchmark.

Pipeline: `text → BGE frozen → fact_emb → Phase 1 (frozen) → sub_kg + routed_alpha → classifier head → softmax over options`.

**Dataset**: SimBench (Hu et al. ICML 2025, arXiv 2510.17516). 7,167 Pop + 6,343 Grouped = ~13.5k samples.

**Pre-req**: 00_setup + 01_phase1 (ph1_v3_minimal 학습 끝나야).

In [ ]:
# Pre-setup — Colab kernel 마다 첫 cell. 이미 mount 됐으면 skip.
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## A. Dataset sanity check

HF `load_dataset` 가 Pop/Grouped schema 다름 으로 fail. pandas 로 직접 load.

In [ ]:
import pandas as pd
from collections import Counter
from phase1.eval_simbench_classifier import parse_simbench_dict

BASE = 'https://huggingface.co/datasets/pitehu/SimBench/resolve/main'
pop = pd.read_csv(f'{BASE}/SimBenchPop.csv')
grouped = pd.read_csv(f'{BASE}/SimBenchGrouped.csv')
print(f'Pop:     {len(pop):>6} rows × {len(pop.columns)} cols')
print(f'Grouped: {len(grouped):>6} rows × {len(grouped.columns)} cols')

for name, df in [('Pop', pop), ('Grouped', grouped)]:
    cnt = Counter()
    for _, row in df.iterrows():
        try: cnt[len(parse_simbench_dict(row['human_answer']))] += 1
        except: cnt['err'] += 1
    print(f'{name} option counts: {dict(sorted(cnt.items(), key=lambda x: (isinstance(x[0], str), x[0])))}')

## B. Train classifier head (~15-20분 첫 실행, ~2분 재실행)

두 번째 실행은 feature cache hit → 빠름.

- **Loss**: KL divergence (predicted softmax || human aggregate, masked invalid options)
- **Eval**: KL + argmax accuracy

In [ ]:
from phase1.eval_simbench_classifier import train_simbench_classifier

result = train_simbench_classifier(
    phase1_run_id='ph1_v3_minimal',
    epochs=20,
    batch_size=256,
    lr=1e-3,
    hidden=512,
    dropout=0.1,
)

## C. 결과 확인

**Paper claim 기준**:
- argmax_acc > 0.30 → paper-worthy (random baseline ~0.25)
- argmax_acc > 0.40 → strong (SimBench paper 의 best LLM = 40.80/100)

In [ ]:
import json
from pathlib import Path

r = json.loads(Path('out/phase1/ph1_v3_minimal/simbench_eval.json').read_text())
print(f'best epoch:    {r["best_epoch"]}')
print(f'best val_kl:   {r["best_val_kl"]:.4f}')
print(f'test kl:       {r["test"]["kl"]:.4f}')
print(f'test acc:      {r["test"]["argmax_acc"]:.4f}')
print(f'test n:        {r["test"]["n"]}')